# Video simulation

This notebook applies the trained **YOLOv11** model from Low-Altitude Smart Rescue Demo to a video, producing a new video with detections drawn as bounding boxes (civilians, rescuers, animals). Useful for testing the model in dynamic scenarios, such as the [firefighters in a flooded area](https://www.youtube.com/watch?v=QnFwDqzCwRU) video used in the project.

---
**Project:** Low-Altitude Smart Rescue Demo

## 0. Dependencies

Install Ultralytics and OpenCV (and tqdm for a progress bar). In Colab, you may need to enable a GPU under *Runtime → Change runtime type*.

In [ ]:
!pip install ultralytics opencv-python tqdm --quiet

## 1. Configuration

Set paths for the **model** (trained weights), **input video**, and **output video**. Adjust `CONFIDENCE` to filter detections (e.g. 0.75 for higher precision).

In [ ]:
from pathlib import Path

MODEL_PATH = "../models/yolov11n/best.pt"
VIDEO_INPUT = "../static/video/rescuer.mp4"
VIDEO_OUTPUT = "detected_output.mp4"
CONFIDENCE = 0.75  # minimum confidence for detection

if not Path(MODEL_PATH).exists():
    raise FileNotFoundError(f"Model not found: {MODEL_PATH}. Train the model or adjust MODEL_PATH.")
if not Path(VIDEO_INPUT).exists():
    raise FileNotFoundError(f"Video not found: {VIDEO_INPUT}. Add the video to the project or adjust VIDEO_INPUT.")
print("Configuration OK.")

## 2. Process video

Loads the model, reads the video frame by frame, runs detection, and writes the result. The progress bar shows progress.

In [ ]:
import cv2
from ultralytics import YOLO

try:
    from tqdm import tqdm
    use_tqdm = True
except ImportError:
    use_tqdm = False

model = YOLO(MODEL_PATH)
cap = cv2.VideoCapture(VIDEO_INPUT)

width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
fps = cap.get(cv2.CAP_PROP_FPS)
total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))

fourcc = cv2.VideoWriter_fourcc(*"mp4v")
out = cv2.VideoWriter(VIDEO_OUTPUT, fourcc, fps, (width, height))

pbar = tqdm(total=total_frames, desc="Processing", unit="frame") if use_tqdm else None
frame_idx = 0
while cap.isOpened():
    ret, frame = cap.read()
    if not ret:
        break
    results = model(frame, conf=CONFIDENCE, verbose=False)
    result_img = results[0].plot()
    out.write(result_img)
    frame_idx += 1
    if pbar:
        pbar.update(1)
    elif frame_idx % 30 == 0:
        print(f"Frames processed: {frame_idx}/{total_frames}")
if pbar:
    pbar.close()
cap.release()
out.release()
print(f"Video saved: {VIDEO_OUTPUT}")

## 3. Play the output video

In [ ]:
from IPython.display import Video
Video(VIDEO_OUTPUT, embed=True, width=640)